In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Load the dataset
train_data_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/bank_customer_churn/train.csv'
train_df = pd.read_csv(train_data_path)

# Display basic information about the dataset
print(train_df.head())
print(train_df.info())


       id  CustomerId     Surname  ...  IsActiveMember EstimatedSalary Exited
0  149380    15780088  Yobachukwu  ...             1.0       103560.98      0
1  164766    15679760    Slattery  ...             0.0       102950.79      0
2  155569    15637678          Ma  ...             0.0       155394.52      0
3  124304    15728693      Galkin  ...             1.0       107428.42      0
4  108008    15613673        Lung  ...             0.0       134110.93      0

[5 rows x 14 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 132027 entries, 0 to 132026
Data columns (total 14 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   id               132027 non-null  int64  
 1   CustomerId       132027 non-null  int64  
 2   Surname          132027 non-null  object 
 3   CreditScore      132027 non-null  int64  
 4   Geography        132027 non-null  object 
 5   Gender           132027 non-null  object 
 6   Age              

In [ ]:


# Check for missing values
print(train_df.isnull().sum())

# Identify categorical and numerical columns
categorical_cols = train_df.select_dtypes(include="object").columns.tolist()
numerical_cols = train_df.select_dtypes(exclude="object").columns.tolist()

# Remove the target column from features
numeric_features = numerical_cols[:-1]  # Exclude 'Exited'
numeric_transformer = StandardScaler()

# Preprocessing for categorical data
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

# Apply transformations
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_cols),
    ]
)

# Split features and target variable
X = train_df.drop(columns=["Exited"])
y = train_df["Exited"]

# Split the data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)



id                 0
CustomerId         0
Surname            0
CreditScore        0
Geography          0
Gender             0
Age                0
Tenure             0
Balance            0
NumOfProducts      0
HasCrCard          0
IsActiveMember     0
EstimatedSalary    0
Exited             0
dtype: int64


In [ ]:



# Define the pipeline
model = Pipeline(steps=[("preprocessor", preprocessor),
                        ("classifier", RandomForestClassifier(random_state=42))])

# Train the model
model.fit(X_train, y_train)

# Predict probabilities on the validation set
y_val_pred_proba = model.predict_proba(X_val)[:, 1]

# Calculate AUC-ROC
roc_auc = roc_auc_score(y_val, y_val_pred_proba)
print(f"AUC-ROC: {roc_auc:.4f}")




AUC-ROC: 0.8786
